In [1]:
import matplotlib.pyplot as plt
import pandas as pd
import re
import os

def parse_champsim_output(file_path):
    """Parses the ChampSim output file to extract relevant metrics."""
    metrics = {}

    if not os.path.exists(file_path):
        print(f"Warning: File {file_path} not found!")
        return metrics

    with open(file_path, 'r') as file:
        for line in file:
            if "cumulative IPC" in line:
                match = re.findall(r"\d+\.\d+", line)
                if match:
                    metrics['IPC'] = float(match[0])
            elif "AVERAGE MISS LATENCY" in line and "L2C" in line:
                match = re.findall(r"\d+\.\d+|\d+", line) 
                if match:
                    metrics['L2C Avg Miss Latency'] = float(match[-1]) 
            elif "L2C PREFETCH" in line and "ACCESS" in line:
                values = re.findall(r"\d+", line) 
                if len(values) >= 4:
                    metrics['L2_PREFETCH Access'] = int(values[-4])  
                    metrics['L2_PREFETCH Hit'] = int(values[-3])    
                    metrics['L2_PREFETCH Miss'] = int(values[-2]) 
            elif "L2C PREFETCH" in line and "REQUESTED" in line:
                values = re.findall(r"\d+", line) 
                if len(values) >= 5:
                    metrics['L2_PREFETCH Requested'] = int(values[-4])  
                    metrics['L2_PREFETCH Issued'] = int(values[-3])    
                    metrics['L2_PREFETCH Useful'] = int(values[-2]) 
                    metrics['L2_PREFETCH Useless'] = int(values[-1])    

    return metrics

In [2]:
prefetchers = ['ip-stride', 'next-line', 'no', 'va-ampm-lite']
bfs_results = {}
dfs_results = {}
spmv_results = {}

for p in prefetchers:
    bfs_results[p] = parse_champsim_output(f'output/{p}/bfs_{p}.txt')
    dfs_results[p] = parse_champsim_output(f'output/{p}/dfs_{p}.txt')
    spmv_results[p] = parse_champsim_output(f'output/{p}/spmv_{p}.txt')

bfs_df = pd.DataFrame(bfs_results)
bfs_df['Benchmark'] = 'bfs'
dfs_df = pd.DataFrame(dfs_results)  
dfs_df['Benchmark'] = 'dfs'
spmv_df = pd.DataFrame(spmv_results)
spmv_df['Benchmark'] = 'spmv'

df = pd.concat([bfs_df, dfs_df, spmv_df])
df_long = df.reset_index().melt(id_vars=['index', 'Benchmark'], var_name='Prefetcher', value_name='Value')
df_long.rename(columns={'index': 'Metric'}, inplace=True)

In [3]:
df_long

,Metric,Benchmark,Prefetcher,Value
0,IPC,bfs,ip-stride,0.9198
1,L2_PREFETCH Access,bfs,ip-stride,3758.0000
2,L2_PREFETCH Hit,bfs,ip-stride,1156.0000
3,L2_PREFETCH Miss,bfs,ip-stride,2602.0000
4,L2_PREFETCH Requested,bfs,ip-stride,2774.0000
...,...,...,...,...
103,L2_PREFETCH Requested,spmv,va-ampm-lite,169508.0000
104,L2_PREFETCH Issued,spmv,va-ampm-lite,15523.0000
105,L2_PREFETCH Useful,spmv,va-ampm-lite,6370.0000
106,L2_PREFETCH Useless,spmv,va-ampm-lite,2586.0000


In [4]:
import plotly.express as px

metrics = df_long['Metric'].unique()

for metric in metrics:
    df_metric = df_long[df_long['Metric'] == metric]

    if "PREFETCH" in metric.upper():
        df_metric = df_metric[df_metric["Prefetcher"] != "no"]
    
    fig = px.bar(
        df_metric, 
        x="Benchmark", 
        y="Value", 
        color="Prefetcher", 
        barmode="group", 
        title=f"{metric} Across Benchmarks",
        labels={"Value": metric, "Benchmark": "Benchmark", "Prefetcher": "Prefetcher"}
    )

    if not df_metric.empty:
        fig.show()